# Stage 0 — Raw data ingestion

Downloads the UCI **ElectricityLoadDiagrams20112014** dataset (370 clients, 15-min consumption, 2011-2014)
and lands it in S3 as partitioned Parquet — our "raw" zone.

Source: https://archive.ics.uci.edu/dataset/321/electricityloaddiagrams20112014
License: CC BY 4.0

**Cost note**: this step is mostly I/O-bound, but the wide→long reshape is memory-hungry
(370 clients x ~140k timestamps melts into ~51M rows if done in one shot). Use `ml.m5.xlarge`
(16GB RAM, ~$0.23/hr, this notebook needs minutes not hours) rather than `ml.t3.medium` (4GB) —
the reshape below is also written to process one year at a time regardless, so peak memory
stays bounded even if you keep `ml.t3.medium`.

In [ ]:
%pip install -q pyarrow boto3 requests tqdm

In [ ]:
import io
import zipfile
from pathlib import Path

import boto3
import pandas as pd
import requests
from tqdm import tqdm

# --- Config ------------------------------------------------------------
DATA_URL = "https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip"
LOCAL_DIR = Path("data_raw")
LOCAL_DIR.mkdir(exist_ok=True)
ZIP_PATH = LOCAL_DIR / "electricity.zip"
TXT_NAME = "LD2011_2014.txt"

BUCKET = "<your-bucket>"
RAW_PREFIX = "ts-forecast-demo/raw/electricity"

s3 = boto3.client("s3")

In [ ]:
def download_file(url: str, dest: Path, chunk_size: int = 1 << 20) -> None:
    """Streams the file to disk with a progress bar (dataset is ~250MB zipped)."""
    if dest.exists():
        print(f"{dest} already exists, skipping download")
        return
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)
                bar.update(len(chunk))


download_file(DATA_URL, ZIP_PATH)

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(LOCAL_DIR)

txt_path = LOCAL_DIR / TXT_NAME
assert txt_path.exists(), f"Expected {txt_path} after extraction"
print(txt_path, txt_path.stat().st_size / 1e6, "MB")

## Parse and reshape

Raw file: semicolon-delimited, decimal comma, one column per client (`MT_001` ... `MT_370`),
first column is the timestamp. We reshape wide -> long (`timestamp, client_id, kw`) which is the
natural tidy format for a multi-series forecasting problem, and partition-friendly for Parquet.

In [ ]:
df_wide = pd.read_csv(
    txt_path,
    sep=";",
    decimal=",",
    index_col=0,
    parse_dates=True,
)
df_wide.index.name = "timestamp"
print(df_wide.shape)  # ~140256 timestamps x 370 clients
df_wide.head()

## Reshape and write, one year at a time

**Why chunked**: melting all 370 clients x ~140k timestamps in one shot builds a ~51M-row
long DataFrame — several GB once you account for pandas' intermediate copies during melt,
sort, and Arrow conversion. That's what triggered the OOM on `ml.t3.medium` (4GB) last time,
with the Jupyter kernel silently killed mid-autosave (no application-level error, since the
OS OOM killer doesn't write to Jupyter's own log stream).

Processing one year at a time bounds peak memory to ~13M rows (one year) instead of ~51M,
and each partition is written and uploaded before the next year is even built.

In [ ]:
import gc

import pyarrow as pa
import pyarrow.parquet as pq

local_out = LOCAL_DIR / "parquet"
local_out.mkdir(exist_ok=True)
client_ids = df_wide.columns.tolist()

for year in sorted(df_wide.index.year.unique()):
    year_slice = df_wide[df_wide.index.year == year]

    df_year = (
        year_slice
        .reset_index()
        .melt(id_vars="timestamp", var_name="client_id", value_name="kw")
    )
    df_year["kw"] = df_year["kw"].astype("float32")
    df_year["kwh"] = df_year["kw"] / 4.0
    df_year["client_id"] = df_year["client_id"].astype("category")
    df_year["year"] = year
    df_year = df_year.sort_values(["client_id", "timestamp"]).reset_index(drop=True)

    part_dir = local_out / f"year={year}"
    part_dir.mkdir(parents=True, exist_ok=True)
    part_file = part_dir / "part-0.parquet"
    pq.write_table(pa.Table.from_pandas(df_year, preserve_index=False), part_file)

    key = f"{RAW_PREFIX}/year={year}/part-0.parquet"
    s3.upload_file(str(part_file), BUCKET, key)
    print(f"year={year}: {len(df_year):,} rows -> s3://{BUCKET}/{key}")

    del df_year, year_slice
    gc.collect()

In [ ]:
# Sanity check: list what landed in S3
resp = s3.list_objects_v2(Bucket=BUCKET, Prefix=RAW_PREFIX)
for obj in resp.get("Contents", [])[:10]:
    print(obj["Key"], obj["Size"])